# Predicting Bladder Cancer Recurrence using the Chimera Challenge Data

### Dataset and Task

This exercise is intended to introduce you to training simple machine learning modeels. We will use a dataset from the plublic "Chimera" challenge (specifically task 3 of the challenge). The Chimera challenge consists of a set of tasks related to predicting therapy responses and outcomes using various types of input data, including radiology images (MRIs), microscopic pathology slides, genetic sequencing data, and other information about patients ("demographic" information such as age and sex and other basic health data such as weight, etc).

At this point I recommend that you take a quick look at the Chimera challenge [website](https://chimera.grand-challenge.org) and read the information about [task 3](https://chimera.grand-challenge.org/ask-3-bladder-cancer-recurrence-prediction/) closely so that you understand the problem you are trying to solve and the sorts of data that you will be working with.

Briefly, this task works with data from patients with a certain type of aggressive bladder cancer called high-risk non-muscle invasive bladder cancer (HR-NMIBC). The idea is take various types of data about the patient and combine it to create a prediction for whether (and how long until) the patient's cancer will recur after the cancer is initially removed and treated. If we could predict this somewhat accurately, it would allow us to use more aggressive treatments for higher risk patients, and monitor them more closely after treatment to check for new tumor growth. Furthermore, it would allow us to gain insights into what factors are important in determining cancer recurrence, and this will inform future research into new treatments.

The task mentions various types of data:

* "Histopathology": This means that a small biopsy was taken from the patients' tumors and imaged at a very high resolution under a powerful microscope to create a digital image in which you can see the individual cells that make up the tumor and even sub-cellular structures such as cell nuclei. You can see an example of one of these images (very zoomed out!) near the top of the task 3 [page](https://chimera.grand-challenge.org/ask-3-bladder-cancer-recurrence-prediction/). These images can tell you a lot about the biology of the tumor, and this information can be used to predict the likelihood of recurrence. In our simplified exercise, we will not work with this data because it is quite challenging to work with these images directly.
* "Clinical data": There are various pieces of clinical information collected during a patient's treatment that may help us predict the likelhood of recurrence. This includes the results of the patient's age, sex, whether they smoke, the stage of the tumor (how advanced it is), the grade of the tumor (how aggressively it is growing), and various other pieces of information. You can read the full list on the task description page. We will be working with this data later.
* "RNA Sequencing": Otherwise known as "transcriptomic" data or abbreviated as "RNASeq", this information is the result of conducting RNA sequencing on the tumor slide to identify and quantify the RNA molecules present. RNA is created in the chain of events that allows cells to create new proteins from the genetic information stored in the cell's DNA, a process known as "transcription". Measuring RNA therefore allows us to measure which genes are actually being expressed, instead of just measuring which genes are present in the DNA. The RNA sequencing data used in this project consists of a large number of measurements (19,359 to be precise) for each patient, where each measurement measures the expression of a particular RNA snippet of interest.

The dataset consists of data from 176 patients, along with information about whether (and when) they experienced recurrence of their tumor.

### Building a Machine Learning Model For Recurrence Prediction

In this exercise, we will be using machine learning methods to train a computer model to that takes in the input data (clinical data and RNA Seq data) and predict whether or not the patient will experience recurrence. Machine learning works by representing the inputs and outputs as a set numbers, then building a mathematical expression that calculates the output numbers from the input numbers. This expression is usually very complicated and involves a lot of parameters. The process of "training" a model involves using optimization methods to find the parameters that best map the input data to the output data. We then hope that this set of parameters works for future patients if we collect their input data and put it into the model.

The type of ML model that we will train is called a "classifier" because it classifies inputs into one of two classes: patients whose cancers do not recur (we will represent this by the output number 0) and patients whose cancers do recur (we will represent this by the output number 1).

Many different types of machine learning models have been developed for classification problems like the one we are trying to solve here. Different models work better than other in different situations, and it is often difficult to know what method will work best for a given problem. That's where you come in! You will conduct experiments with various different types of ML model and compare their performance. We will then write up a short report detailing your experimental findings.

### Python Libraries

Throughout this exercise you will make use of a few different open-source Python "libraries". Open-source libraries are packages of software that are free for you to download and use in your own software. As you get more comfortable with the exercise, you could consult the documentation of each library to understand what variations are possible, and use that information to make changes to the code below to investigate the effect on performance.

These are the most important libraries that you will use:

* [pandas](https://pandas.pydata.org/docs/reference/index.html): This is a library for working with "data frames". Data frames are basically a way of representing a table of data in a computer, a bit like a spreadsheet in Microsoft Excel or Google Sheets.
* [scikit-learn](https://scikit-learn.org/stable/): Scikit-learn is a library that implements many different machine learning algorithms (in this library, machine learning models are called "estimators"). It also has other useful things like functions to calculate metrics of model performance. Later on, you may like to read the [getting started](https://scikit-learn.org/stable/getting_started.html) guide if you want to play around with building other types of machine learning model ("estimator").

### How To Complete This Exercise

You should work through each "cell" (box) and execute the code to load in the data and train various models. You do not need to understand what every line of code is doing, but read the comments above them for an explanation. Many some cells you should just leave as they are, but in some cells you will be told to change some parts of the code and see the results.

**Remember to record the results of each experiment you run**. You will need these results to write up your experiments at the end.

When the exercise is complete, you should write up a brief report of your results in the style of a scientific paper. It should be around 2-4 pages in length. The report should have the following sections:

* *Introduction*: One or two paragraphs explaining what you are investigating and why it is important
* *Methods*: Describe the methods that you used to investigate.
* *Results*: Present the results of your experiments in tables and/or plots.
* *Conclusion*: One or two paragraphs about what you conclude from your experimental results.

Okay, now to the code...

# Imports

First we need to import the Python libraries that we are going to use so that they are available later.

In [ ]:
import pathlib
import json
import pandas
import sklearn  # this is how the scikit-learn is referred to within Python
import numpy as np

# Loading the Clinical Data

The clinical data are stored in a set of "JSON" files, with one file per patient. The code in the following cell simply loads in all the files for each patient and constructs a pandas Data Frame (like a table) containing all the data that we will use later.

In [ ]:
data_path = pathlib.Path("/autofs/space/crater_001/datasets/public/chimera/task3/data")

all_patient_data = []
for patient_dir in data_path.iterdir():  # loop over all the folders ("directories") containing the data
    if not patient_dir.is_dir():
        # Skip other files such as the quality control CSV
        continue

    patient_id = patient_dir.name
    with patient_dir.joinpath(f"{patient_id}_CD.json").open("r") as jf:
        patient_data = json.load(jf)

    patient_data["patient_id"] = patient_id
    all_patient_data.append(patient_data)

clinical_data_frame = pandas.DataFrame(all_patient_data).sort_values("patient_id")

If you execute the next cell, you should see the table containing the clinical data. You should see that each row contains the information about one patient, and each column contains a particular piece of information about each patient. When used for machine learning, this pieces of information are called "features" and the patients are called "samples", so the dataframe consists of one row per sample and one column per feature:

In [ ]:
clinical_data_frame

### Transforming the Clinical Data For Use With Machine Learning

Remember that machine learning represents all the input and output data as numbers. But you should see that the above table contains text ("strings" as we call them in programming). The next steps take the above data frame and tweak it slightly to represent the same information in a way that can be used for machine learning.

In [ ]:
# Remove the tumor column as it is useless (only one unique value: "primary")
preprocessed_clinical_data = clinical_data_frame.drop("tumor", axis=1)
# We also do not use the time to recurrence column
preprocessed_clinical_data = preprocessed_clinical_data.drop("time_to_HG_recur_or_FUend", axis=1)

# For all binary columns, replace with 0 or 1 by testing for equality
for column, positive_value in [
    ("sex", "Male"),
    ("stage", "TaHG"),
    ("grade", "G3"),
    ("reTUR", "Yes"),
    ("variant", "UCC + Variant"),
    ("EORTC", "Highest risk"),
]:
    preprocessed_clinical_data[column] = (preprocessed_clinical_data[column] == positive_value)

# For other categorical columns, use get_dummies for a one-hot encoding
preprocessed_clinical_data = pandas.get_dummies(
    preprocessed_clinical_data,
    columns=["smoking", "substage", "BRS", "LVI"],
)

# For numerical columns, divide by the maximum value
for column in ["no_instillations", "age"]:
    preprocessed_clinical_data[column] = preprocessed_clinical_data[column] / preprocessed_clinical_data[column].max()

Now look at the `preprocessed_clinical_data` by running the next cell. You will see that (apart from the ``patient_id`` column that we will ignore later), everything is represented as a number or as ``False`` or ``True``. Scikit-learn knows to treat ``False`` and ``True`` as the numbers 0 and 1 respecitively.

In [ ]:
preprocessed_clinical_data

### Splitting the Data For Training and Testing

How will we know how accurate our model is once it is trained?

We could just look at how accurate is on the training data, but there is a big problem here. Models are often much more accurate on the data they were trained on than other data (which makes sense, since the model parameters were optimized specifically for that data). To get around this, we use a process called *cross-validation*. In cross-validation, we "hold aside" a small fraction of the data to use and do not use it for training the model. We then use this *test set* at the end to see how accurate the model is. This gives us a much better idea of how likely the model is to be going forward.

In the next cell, we divide the whole dataset into two sets: training and testing. We will use 20% of the dataset for testing and the remaining 80% for training. To keep things simple, we will just take every 5th row and use that for testing.

In [ ]:
training_data = preprocessed_clinical_data[preprocessed_clinical_data.index % 5 > 0]  # rows that are not divisble by 5
test_data = preprocessed_clinical_data[preprocessed_clinical_data.index % 5 == 0]  # rows that are divisble by 5
print("There are", len(training_data), "rows in the training data")
print("There are", len(test_data), "rows in the test data")

# Training A Simple Classifier

Now we will move on to actually training a simple classifier.

The first step is to split both the training and testing data into four arrays of numbers:

* One array using the multiple *input* columns that are passed to the model's input.
* One array using the single *label* column that gives the desired model output, used to train the model.

In [ ]:
input_columns = [c for c in training_data.columns if c not in ("patient_id", "progression")]
label_column = "progression"
print("The following columns are used for input:", input_columns)

clinical_train_input = training_data[input_columns].values.astype(float)
train_labels = training_data[label_column].values

clinical_test_input = test_data[input_columns].values.astype(float)
test_labels = test_data[label_column].values

The first model that we will start with will be a logistic regression model. The equation of this model is actually very straightforward. First we add up all of the inputs $x_i$, multiplied by a parameter per input $c_i$:

$$
y = \sum_i c_i x_i
$$

Then we pass this through a special function called the *logistic function*, which takes in $y$ (which can have any value) and outputs a number, $z$, in the range 0 to 1.

$$
z = \frac{1}{1 + e^{-y}} 
$$

The output represents a predicted probability of the case being positive (which for us means recurrence). So if $z=0.0$, the model is certain that the patient will not recur, if $z=1.0$, the model is certain that the patient will recur, if $z=0.5$, the model thinks the patient is equally likely to recur or not recur.

The following lines of code first create a classifier *object*, then use the *fit* method to fit the classifier's parameters (the $c_i$ in the equations above) to the training inputs and training labels. *Fit* is just another word for *train*.

In [ ]:
logistic_classifier = sklearn.linear_model.LinearRegression()
logistic_classifier.fit(clinical_train_input, train_labels)

Now that we have the trained (fitted) model, we can use this model to make predictions about the test inputs.

In [ ]:
test_predictions = logistic_classifier.predict(clinical_test_input)

Let's look at these predictions that we just calculated:

In [ ]:
print(test_predictions)

As expected, they are all numbers between 0.0 and 1.0, showing how likely think the model thinks each patient is to recur.

### Measuring Classifier Performance

Now we have the model outputs on the test set, we can compare them to the labels (the known recurrence values), to assess how accurately the model performs.

The simplest metric we can use is *accuracy*, this simply means what proportion of the predictions did the model get right? In order to do this, we first threshold the model outputs:

* If the model's prediction is less than 0.5 (meaning it thinks the patient is unlikely to recur), we consider the prediction to be negative (0).
* If the model's prediction is greater than 0.5 (meaning it thinks the patient is likely to recur), we consider the prediction to be positive (1).

Sci-kit learn's accuracy score function handles this for us if we pass it the model's predictions compared to 0.5, and the true labels:


In [ ]:
accuracy = sklearn.metrics.accuracy_score(y_pred=test_predictions > 0.5, y_true=test_labels)
print(accuracy)

Another metric that we often use is called the AUROC (or ROC AUC) score. While the definition of this metric is a little complicated, it is basically a number between 0.0 and 1.0 that describes how well separated the model's predictions the positive samples (those that recur) from the negative samples (those that do not recur). A model whose guesses are random should get an AUROC of 0.5 (intuitively if you randomly guess, you will be right half the time). Therefore, we are hoping for a score in the range 0.5 to 1.0, where 1.0 means perfect classification.

Again, Sci-kit Learn has a function that calculations the AUROC for us:

In [ ]:
auroc = sklearn.metrics.roc_auc_score(y_score=test_predictions, y_true=test_labels)
print(auroc)

### Examining Importance of Features

We have a classifier that is performing okay. But how do we know how it is making its predictions? We can examine the coefficients ($c_i$) of the fitted model. If a coefficient is large, it means that that feature is very important in the decision process. If it is close to zero, it means the feature is not important. If it is very large and negative, it means it is very important, but in the opposite direction to a large positive value, i.e. it is strongly predictive of no recurrence rather than predictive of recurrence.

The next cell looks at the coefficients of the model, and relates them to the feature column names that they represent. Which are the most important features? Referring back to the description of the features on the task description page, try to express in words what this model is telling us:

In [ ]:
for column_name, coefficient in zip(input_columns, logistic_classifier.coef_):
    print(column_name, coefficient)

### Balanced Class Weights

If you look at the number of cases that do and do not recur in the training set, you will see that the dataset is quite *imbalanced*, i.e. there are more caes that do not recur/progress than cases that do:

In [ ]:
training_data["progression"].value_counts()

This is likely to adversely affect the model fitting. To counter this, we can use the ``class_weight`` parameter of the ``LogisticRegression`` class, and set it to ``"balanced"``. Run the fitting and evaluation again with this parameter set, and see whether this improves the performance:

In [ ]:
logistic_classifier = sklearn.linear_model.LogisticRegression(class_weight="balanced")
logistic_classifier.fit(clinical_train_input, train_labels)
test_predictions = logistic_classifier.predict(clinical_test_input)
accuracy = sklearn.metrics.accuracy_score(y_pred=test_predictions > 0.5, y_true=test_labels)
print("accuracy:", accuracy)
auroc = sklearn.metrics.roc_auc_score(y_score=test_predictions, y_true=test_labels)
print("auroc:", auroc)

# Exploring Other Classifiers On The Clinical Data

We have so far looked at a single classifier method: logistic regression. But there are many other methods implemented in Scikit-learn to try and improve upon this "baseline" result:

- [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html): Similar to the basic Logistic Regression model, but includes a penalty on the coefficients becoming too large, which should help the model generalize better to new data.
- [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier): This model consists of a "forest" (just a collection) of a large number of decision trees. Each decision tree is like a flow diagram, where each node asks questions about the data, and sends the data to one or another node depending on the answer. Important parameters include: ``n_estimators`` the number of trees in the forest (try increasing this from the default of 100 to see whether that improves performance), ``max_depth`` the maximum number of decision nodes per tree (try setting this to a relatively small number such as 3, 4, or 5 because smaller simpler models may perform on the test data).
- [GradientBoostingClassifier]([https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html]): A variant on random forests that uses simpler trees trained in a different way. Important parameters include: ``n_estimators`` and ``max_depth`` (like random forests), ``subsample`` (try setting this to a number less than 1.0 to train each tree on a subset of the training data).
- [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html): An algorithm like random forests, that concentrates the training of later trees on data that have been misclassified by earlier trees. Again, try the ``n_estimators`` parameter.
- [GaussianNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB): A simple model that considers each feature to belong to a normal distribution for each class to derive an optimal classifier.
- [SVC (Support Vector Classifier)](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC): This model works in a totally different way. It basically finds a set of training samples that are representative of each class (recurrence or no recurrence), and measures the distance of a point from each one. Important parameters: ``C``, try increasing this number to give a simpler model that may do better on the training data, ``kernel`` this is basically a function used to measure distance, try ``sigmoid``, ``poly`` and ``linear`` in addition to the default ``rbf``.

Adapt the following code (by replacing the LogisticRegression model with the relevant classifer) to try each of these types of model. For the most promosing models (with the best performance), try changing some of the recommended parameters (consult the linked documentation for more details), to see whether you can improve upon the result even further.

**Remember to record all your results!**

In [ ]:
# Create the classifier
classifier = sklearn.linear_model.LinearRegression()  # change this, and add parameters here

# Fit/train the classifier
classifier.fit(clinical_train_input, train_labels)

# Make predictions on the test set
test_predictions = classifier.predict(clinical_test_input)

# Compute accuracy metrics
accuracy = sklearn.metrics.accuracy_score(y_pred=test_predictions > 0.5, y_true=test_labels)
print("accuracy:", accuracy)
auroc = sklearn.metrics.roc_auc_score(y_score=test_predictions, y_true=test_labels)
print("auroc:", auroc)

**Bonus Task:** Some of these methods have better ways of examining feature importance. In particular the `RandomForestClassifier` has the `classifier.feature_importances_` attribute that will tell you which features are important. When you train a random forest classifier, check this attribute, and see whether the most important features agree with those for the logistic regression model that we looked at above.

# Loading the RNASeq Data

The RNA sequencing data is also stored in per-patient JSON files. The following code cell loads in the RNA data and makes it into a data frame.

In [ ]:
all_patient_data = []
for patient_dir in data_path.iterdir():  # loop over all the folders ("directories") containing the data
    if not patient_dir.is_dir():
        # Skip other files such as the quality control CSV
        continue

    patient_id = patient_dir.name
    with patient_dir.joinpath(f"{patient_id}_RNA.json").open("r") as jf:
        patient_data = json.load(jf)

    patient_data["patient_id"] = patient_id
    all_patient_data.append(patient_data)

rna_data_frame = pandas.DataFrame(all_patient_data).sort_values("patient_id")

Let's take a look at this new data frame. Remember this is *much* bigger than the clinical data frame because it has nearly 20,000 measurements (columns) for each patient. 

In [ ]:
rna_data_frame

Let's split the data frame into train and test, and inputs and labels, as we did before. The labels are the same as before, so we can re-use the arrays that we previously calculated:

In [ ]:
rna_training_data = rna_data_frame[rna_data_frame.index % 5 > 0]  # rows that are not divisble by 5
rna_test_data = rna_data_frame[rna_data_frame.index % 5 == 0]  # rows that are divisble by 5

rna_input_columns = [c for c in rna_data_frame.columns if c != "patient_id"]

rna_train_input = rna_training_data[rna_input_columns].values.astype(float)
rna_test_input = rna_test_data[rna_input_columns].values.astype(float)

### Exploring Models For RNA Seq Prediction Models

Repeat the above experiment to train and compare models prediction of recurrence from RNA input data. Adapt the code snippets above into the cells below to train and compare models using the `rna_train_input` with the existing `train_labels`, and evaluate them using the `rna_test_input` with the `test_labels`.

In [ ]:
# Fill in this code block!

### Feature Selection

You will probably find that these models do not work very well. This is because there are too many features (nearly 20,000) for the model to choose from. The following code snippet creates a reduced set of features that pre-selects the best features. Training a model with a smaller number of useful features (and eliminating all the irrelevant feautures) often contributes leads to a better optimization process and a better model.

In [ ]:
# Use only the training data to find the best features
feature_set_reducer = sklearn.feature_selection.SelectKBest(
    sklearn.feature_selection.f_classif,
    k=100  # 'k' here controls now many features will be selected
)
feature_set_reducer.fit_transform(rna_train_input, train_labels)

# Then apply this same feature set reduction to the train and test data
reduced_rna_train_input = feature_set_reducer.transform(rna_train_input)
reduced_rna_test_input = feature_set_reducer.transform(rna_test_input)

Repeat your above experiments using the reduced feature set for both training and testing (i.e. `reduced_rna_train_input` and `train_labels` for training, and `reduced_rna_test_input` and `test_labels` for testing). Does this improve performance? Also, vary the value for `k` to select more features or fewer features, how does this affect performance?

In [ ]:
# Fill in code in this block to train models using only the selected RNA features

### Combining RNA and Clinical Data Into a Single Model

We have trained models separately on the clinical data and the RNA Seq. Now let's combine to create a single model. The first step is to combine the arrays of features from the clinical data and (reduced) feautures from the RNA. The arrays are numpy arrays (a big 2D array of numbers). You can combine two numpy arrays horizontally with this function: `combined_data = np.hstack([array1, array2])` or vertically with `combined_data = np.vstack([array1, array2])`. Which is the correct way to combine them? Think about whether the rows and columns represent the features and samples. Are we trying to combine features, or samples? Remember to combine both the training data and test data.

In [ ]:
# Fill in this code block to combine the RNA and clinical
# data input arrays (for both training and testing data)
# into combined arrays for training a combined model

Now try to train a combined model using the same methods as above. Does this improve performance?